# Speech Emotion Recognition using MFCC and CNN
SRM Institute of Science and Technology, Kattankulathur
Undergraduate Research Opportunities Programme (UROP)

---

## Abstract
This project builds a Speech Emotion Recognition system that classifies human speech audio into six emotion categories using deep learning. Audio features are extracted using Mel-Frequency Cepstral Coefficients (MFCCs) along with their delta and delta-delta derivatives, and fed into a Convolutional Neural Network (CNN) for classification. The model is trained on the RAVDESS dataset with data augmentation to improve generalisation.

Emotions classified: angry, calm, fearful, happy, neutral, sad

---

## Table of Contents
1. Setup and Imports
2. Dataset Overview
3. Feature Extraction
4. Data Preparation and Train/Test Split
5. CNN Model Architecture
6. Training
7. Results and Visualisation
8. Conclusion

---
## 1. Setup and Imports

In [ ]:
import os
import json
import random
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dense, Dropout, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Librosa    : {librosa.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 2. Dataset Overview

### RAVDESS - Ryerson Audio-Visual Database of Emotional Speech and Song

Source: Livingstone and Russo (2018), PLOS ONE. https://doi.org/10.1371/journal.pone.0196391

- 24 professional actors (12 male, 12 female)
- Around 2068 audio samples across 6 emotion classes
- All files are 16-bit, 48kHz .wav format
- Audio files are organised into per-emotion subfolders under dataset/

Note: Neutral has fewer samples than other emotions because RAVDESS only records neutral speech at one intensity level. This is expected and does not indicate a data error.

In [ ]:
DATASET_PATH = "dataset"

emotion_counts = {}
for emotion in sorted(os.listdir(DATASET_PATH)):
    folder = os.path.join(DATASET_PATH, emotion)
    if os.path.isdir(folder):
        count = len([f for f in os.listdir(folder) if f.endswith(".wav")])
        emotion_counts[emotion] = count

print("Samples per emotion:")
total = 0
for emotion, count in emotion_counts.items():
    print(f"  {emotion:<10}: {count}")
    total += count
print(f"\n  TOTAL     : {total} audio files")

plt.figure(figsize=(9, 5))
colors = ['#e74c3c','#3498db','#9b59b6','#2ecc71','#95a5a6','#f39c12']
bars = plt.bar(emotion_counts.keys(), emotion_counts.values(),
               color=colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, emotion_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             str(val), ha='center', va='bottom', fontsize=10)
plt.title("Dataset - Emotion Class Distribution", fontsize=14, fontweight='bold')
plt.xlabel("Emotion", fontsize=12)
plt.ylabel("Number of Samples", fontsize=12)
plt.ylim(0, max(emotion_counts.values()) + 40)
plt.tight_layout()
plt.savefig("outputs/dataset_distribution.png", dpi=150)
plt.show()

In [ ]:
# Waveform and MFCC sample for each emotion
SAMPLE_RATE = 22050

fig, axes = plt.subplots(len(emotion_counts), 2, figsize=(14, 3 * len(emotion_counts)))

for i, emotion in enumerate(sorted(emotion_counts.keys())):
    folder = os.path.join(DATASET_PATH, emotion)
    sample_file = os.path.join(folder, os.listdir(folder)[0])
    signal, sr = librosa.load(sample_file, sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40)

    axes[i, 0].plot(signal, color='steelblue', linewidth=0.5)
    axes[i, 0].set_title(f"{emotion.capitalize()} - Waveform", fontsize=10)
    axes[i, 0].set_xlabel("Samples")
    axes[i, 0].set_ylabel("Amplitude")

    img = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[i, 1])
    axes[i, 1].set_title(f"{emotion.capitalize()} - MFCC", fontsize=10)
    fig.colorbar(img, ax=axes[i, 1])

plt.suptitle("Waveform and MFCC per Emotion", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("outputs/waveform_mfcc_samples.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Feature Extraction - MFCC + Delta + Augmentation

### Why Delta MFCCs?

Standard MFCCs capture static spectral features of speech. Adding delta (first derivative) and delta-delta (second derivative) coefficients captures how the speech signal changes over time. This is important for emotion recognition because emotions like angry and fearful may look similar in a static snapshot but differ significantly in their dynamics.

The three feature maps are stacked as channels, giving an input shape of (40, 174, 3) instead of (40, 174, 1).

### Why Data Augmentation?

With around 2068 samples across 6 classes, the model tends to overfit. Three augmentation techniques are applied to the training data only:

- Gaussian noise: simulates background noise conditions
- Time stretching: slightly speeds up or slows down the speech
- Pitch shifting: shifts pitch by a few semitones

This gives roughly 4x more training samples without collecting new recordings.

In [ ]:
SAMPLE_RATE = 22050
N_MFCC      = 40
MAX_LEN     = 174

def extract_features(signal, sr):
    mfcc   = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=N_MFCC)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    def pad_or_trim(m):
        if m.shape[1] < MAX_LEN:
            return np.pad(m, ((0, 0), (0, MAX_LEN - m.shape[1])))
        return m[:, :MAX_LEN]

    return np.stack([pad_or_trim(mfcc), pad_or_trim(delta), pad_or_trim(delta2)], axis=-1)


def augment_signal(signal, sr):
    return [
        signal + 0.005 * np.random.randn(len(signal)),
        librosa.effects.time_stretch(signal, rate=1.1),
        librosa.effects.pitch_shift(signal, sr=sr, n_steps=2)
    ]


X, y = [], []
errors = 0

for emotion in sorted(os.listdir(DATASET_PATH)):
    emotion_path = os.path.join(DATASET_PATH, emotion)
    if not os.path.isdir(emotion_path):
        continue

    files = [f for f in os.listdir(emotion_path) if f.endswith(".wav")]
    print(f"Processing {emotion:<10} - {len(files)} files", end="")

    for file in files:
        try:
            signal, sr = librosa.load(os.path.join(emotion_path, file), sr=SAMPLE_RATE)
            X.append(extract_features(signal, sr))
            y.append(emotion)
            for aug in augment_signal(signal, sr):
                X.append(extract_features(aug, sr))
                y.append(emotion)
        except Exception as e:
            errors += 1

    print(f"  ->  {len([i for i in y if i == emotion])} total (with augmentation)")

X = np.array(X)
y = np.array(y)

print(f"\nExtraction complete.")
print(f"  Feature matrix shape : {X.shape}")
print(f"  Total samples        : {len(y)}")
print(f"  Errors skipped       : {errors}")

---
## 4. Data Preparation and Train/Test Split

- Labels are integer encoded using LabelEncoder
- Features are normalised per channel (zero mean, unit variance)
- Data is split 80% training and 20% testing with stratify to keep class balance equal in both sets

In [ ]:
le            = LabelEncoder()
y_encoded     = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)
NUM_CLASSES   = len(le.classes_)

print("Emotion classes :", list(le.classes_))
print("Number of classes:", NUM_CLASSES)

for c in range(X.shape[-1]):
    X[..., c] = (X[..., c] - np.mean(X[..., c])) / (np.std(X[..., c]) + 1e-8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTrain samples : {X_train.shape[0]}  (80%)")
print(f"Test  samples : {X_test.shape[0]}  (20%)")
print(f"Input shape   : {X_train.shape[1:]}")

---
## 5. CNN Model Architecture

The input (40 x 174 x 3) is treated as a multi-channel image and passed through a 4-block CNN. Each block contains a Conv2D layer for feature extraction, BatchNormalization for training stability, MaxPooling2D for downsampling, and Dropout for regularisation. The classifier head uses two Dense layers ending with a softmax output over 6 emotion classes.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=X_train.shape[1:]),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## 6. Training

Two callbacks are used during training:

- EarlyStopping: stops training if validation loss does not improve for 12 consecutive epochs and restores the best weights
- ReduceLROnPlateau: reduces the learning rate by half if validation loss plateaus for 6 epochs, down to a minimum of 0.000001

In [ ]:
EPOCHS     = 80
BATCH_SIZE = 32

callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

os.makedirs("outputs", exist_ok=True)
model.save("outputs/saved_model.keras")
np.save("outputs/X_test.npy", X_test)
np.save("outputs/y_test.npy", y_test)
np.save("outputs/label_classes.npy", le.classes_)
with open("outputs/history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy : {acc * 100:.2f}%")
print(f"Final Test Loss     : {loss:.4f}")

---
## 7. Results and Visualisation

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2, linestyle='--')
axes[0].set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(history.history['loss'],     label='Train', linewidth=2, color='coral')
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2, linestyle='--', color='darkred')
axes[1].set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("outputs/training_curves.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
cm     = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Speech Emotion Recognition', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Emotion', fontsize=11)
plt.ylabel('True Emotion', fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Classification report
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))

In [ ]:
# Per class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(9, 5))
bars = plt.bar(le.classes_, per_class_acc * 100,
               color=['#e74c3c','#3498db','#9b59b6','#2ecc71','#95a5a6','#f39c12'],
               edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val*100:.1f}%", ha='center', va='bottom', fontsize=10)
plt.axhline(y=np.mean(per_class_acc)*100, color='black', linestyle='--',
            linewidth=1.2, label=f'Average: {np.mean(per_class_acc)*100:.1f}%')
plt.title('Per Class Accuracy', fontsize=13, fontweight='bold')
plt.xlabel('Emotion', fontsize=11)
plt.ylabel('Accuracy (%)', fontsize=11)
plt.ylim(0, 115)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/per_class_accuracy.png", dpi=150)
plt.show()

In [ ]:
# Misclassification chart - which emotions get confused with each other
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
np.fill_diagonal(cm_norm, 0)  # remove correct predictions to highlight errors

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Misclassification Rates - Where the Model Gets Confused', fontsize=12, fontweight='bold')
plt.xlabel('Predicted as', fontsize=11)
plt.ylabel('Actual Emotion', fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/misclassification_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Confidence distribution - how confident was the model across all predictions
probs_all   = model.predict(X_test)
max_probs   = np.max(probs_all, axis=1)
correct     = (y_pred == y_true)

plt.figure(figsize=(10, 5))
plt.hist(max_probs[correct],  bins=30, alpha=0.7, color='#2ecc71', label='Correct predictions')
plt.hist(max_probs[~correct], bins=30, alpha=0.7, color='#e74c3c', label='Wrong predictions')
plt.title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Confidence (max softmax probability)', fontsize=11)
plt.ylabel('Number of Samples', fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/confidence_distribution.png", dpi=150)
plt.show()

print(f"Average confidence on correct predictions  : {max_probs[correct].mean()*100:.1f}%")
print(f"Average confidence on wrong predictions    : {max_probs[~correct].mean()*100:.1f}%")

In [ ]:
# Prediction probabilities for 3 random test samples
sample_indices = random.sample(range(len(X_test)), 3)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, idx in zip(axes, sample_indices):
    probs  = probs_all[idx]
    colors = ['#e74c3c' if i == np.argmax(probs) else '#3498db' for i in range(NUM_CLASSES)]
    ax.bar(le.classes_, probs, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f"True: {le.classes_[y_true[idx]]}\nPred: {le.classes_[np.argmax(probs)]}", fontsize=10)
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle("Predicted Probabilities - 3 Random Test Samples", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/sample_predictions.png", dpi=150)
plt.show()

---
## 8. Conclusion

### Summary

| Component | Detail |
|-----------|--------|
| Dataset | RAVDESS (combined, around 2068 original samples) |
| Emotions | angry, calm, fearful, happy, neutral, sad |
| Features | MFCC + delta + delta-delta (40 coefficients, 3 channels) |
| Augmentation | gaussian noise, time stretch, pitch shift (4x samples) |
| Model | 4-block CNN with Dense classifier |
| Train/Test split | 80% / 20% stratified |
| Optimiser | Adam with ReduceLROnPlateau |
| Regularisation | Dropout, BatchNormalization, EarlyStopping |

### Observations

Delta and delta-delta features capture how speech changes over time, which is important for distinguishing emotions that sound similar in static snapshots. Data augmentation significantly reduced the overfitting gap between training and validation accuracy. Neutral has fewer samples than other classes because RAVDESS only records neutral speech at one intensity level, which is standard for this dataset.

### Future Work

- Record real-world speech samples across different environments for better generalisation
- Experiment with Mel-spectrogram and Chroma features alongside MFCCs
- Explore LSTM and Transformer architectures for modelling longer temporal patterns
- Extend the system to handle more emotion classes such as disgust and surprise

### References

Livingstone S, Russo F (2018). The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS). PLOS ONE 13(5): e0196391. https://doi.org/10.1371/journal.pone.0196391